# Notebook 4: Llama teacher soft labels on the SNLI 100K training set
**Kaggle: GPU T4, Internet ON, Secret `HF_TOKEN`, attach Datasets `nli-data` AND the Dataset made from Notebook 3's output (it must contain the folder `llama1b_lora_adapter`).**

The fine-tuned Llama-3.2-1B-LoRA (Notebook 3) scores each SNLI training pair. We save the raw logits of the three label tokens
(`z_ent, z_neu, z_con`), so Notebook 6 can apply any temperature. Same prompt and label tokens as Notebook 3.

Sanity check built in: the adapter must reproduce its saved SNLI-val accuracy (about 91.98%); if not, the adapter did not load correctly.
Caveat for the report: the teacher was trained on these same 100K pairs, so its labels on them are in-sample (sharper than on unseen data).
Expected runtime: about 25 minutes.

In [1]:
import glob, json, os, subprocess, sys, time
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)   # Kaggle's old torchao breaks peft
from pathlib import Path
import numpy as np, pandas as pd, torch
try:
    import peft
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft"], check=True); import peft
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

hits = glob.glob(os.environ.get("NLI_ROOT", "/kaggle/input") + "/**/nli_common.py", recursive=True)
assert hits, "attach the 'nli-data' Dataset"
ROOT = Path(hits[0]).parent; sys.path.insert(0, str(ROOT)); DATA = str(ROOT / "data")
from nli_common import load_split

ad = glob.glob(os.environ.get("NLI_ROOT", "/kaggle/input") + "/**/llama1b_lora_adapter/adapter_config.json", recursive=True)
assert ad, "attach the Dataset containing Notebook 3's output (folder llama1b_lora_adapter)"
ADAPTER = str(Path(ad[0]).parent)
OUT = Path(os.environ.get("NLI_OUT", "/kaggle/working/teacher")); OUT.mkdir(parents=True, exist_ok=True)
assert torch.cuda.is_available(), "turn on the GPU"
dev = "cuda"
print("GPU:", torch.cuda.get_device_name(0), "| adapter:", ADAPTER)

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
assert HF_TOKEN, "add the Kaggle Secret HF_TOKEN"

MODEL = "meta-llama/Llama-3.2-1B-Instruct"
MAX_LEN = 512
train = load_split("train_100k", DATA)
val = load_split("snli_val", DATA)

GPU: Tesla T4 | adapter: /kaggle/input/datasets/punyapratap07/nli-llama-adapter/results/llama1b_lora/llama1b_lora_adapter


## Prompt and helpers (identical to Notebook 3)

In [2]:
tok = AutoTokenizer.from_pretrained(MODEL, token=HF_TOKEN)
if "<|finetune_right_pad_id|>" in tok.get_vocab():
    tok.pad_token = "<|finetune_right_pad_id|>"
elif tok.pad_token is None:
    tok.pad_token = tok.eos_token
PAD = tok.pad_token_id
LABEL_WORDS = [" entailment", " neutral", " contradiction"]
LABEL_IDS = [tok(w, add_special_tokens=False).input_ids[0] for w in LABEL_WORDS]
assert LABEL_IDS == [87092, 21277, 50859], f"label token ids differ from Notebook 3: {LABEL_IDS}"
LABEL_IDS_T = torch.tensor(LABEL_IDS, device=dev)

def build_prompt(premise, hypothesis):      # MUST match Notebook 3
    return (f"Premise: {premise}\nHypothesis: {hypothesis}\n"
            "Does the premise entail the hypothesis, contradict it, or is it neutral?\n"
            "Answer with one word: entailment, neutral, or contradiction.\nAnswer:")

def encode_df(df):
    prompts = [build_prompt(p, h) for p, h in zip(df["premise"], df["hypothesis"])]
    seqs = tok(prompts, add_special_tokens=True).input_ids
    for i, s in enumerate(seqs):
        words = df["premise"].iloc[i].split()
        while len(s) > MAX_LEN and len(words) > 5:
            words = words[: int(len(words) * 0.85)]
            s = tok(build_prompt(" ".join(words), df["hypothesis"].iloc[i]), add_special_tokens=True).input_ids
        seqs[i] = s
    return seqs

def pad_left(seqs):
    L = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), L), PAD, dtype=torch.long); mask = torch.zeros((len(seqs), L), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, L - len(s):] = torch.tensor(s); mask[i, L - len(s):] = 1
    pos = (mask.cumsum(1) - 1).clamp(min=0)
    return ids.to(dev), mask.to(dev), pos.to(dev)

_KEEP = {"name": "logits_to_keep"}
def last_logits(model, ids, mask, pos):
    while True:
        try:
            kw = {_KEEP["name"]: 1} if _KEEP["name"] else {}
            return model(input_ids=ids, attention_mask=mask, position_ids=pos, **kw).logits[:, -1, :]
        except TypeError:
            if _KEEP["name"] is None: raise
            _KEEP["name"] = "num_logits_to_keep" if _KEEP["name"] == "logits_to_keep" else None

@torch.no_grad()
def label_logits(model, df, bs=64, log_every=100):
    """Raw logits of the 3 label tokens, shape (N, 3), original row order."""
    model.eval()
    seqs = encode_df(df)
    order = np.argsort([len(s) for s in seqs])
    out = np.zeros((len(df), 3), dtype=np.float32); t0 = time.time()
    for b, s in enumerate(range(0, len(order), bs)):
        idx = order[s:s + bs]
        ids, mask, pos = pad_left([seqs[j] for j in idx])
        with torch.autocast("cuda", dtype=torch.float16):
            lg = last_logits(model, ids, mask, pos)
        out[idx] = lg[:, LABEL_IDS_T].float().cpu().numpy()
        if (b + 1) % log_every == 0:
            done = min(s + bs, len(order)); el = time.time() - t0
            print(f"  {done:,}/{len(order):,}  {done / el:.0f} ex/s  ETA {(len(order) - done) / (done / el) / 60:.1f} min")
    return out

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

## Load the teacher and check it against the saved Notebook 3 result

In [3]:
try:
    base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, token=HF_TOKEN)
except TypeError:
    base = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16, token=HF_TOKEN)
model = PeftModel.from_pretrained(base.to(dev), ADAPTER, is_trainable=False)
model.eval()

zv = label_logits(model, val)
val_acc = float((zv.argmax(1) == val["label"].to_numpy()).mean())
print(f"SNLI val accuracy of loaded teacher: {val_acc:.4f}  (Notebook 3 reported 0.9198)")
assert abs(val_acc - 0.9198) < 0.01, "adapter does not reproduce Notebook 3's validation accuracy: wrong or unloaded adapter"

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

  6,400/9,842  89 ex/s  ETA 0.6 min
SNLI val accuracy of loaded teacher: 0.9198  (Notebook 3 reported 0.9198)


## Label the 100K training set

In [4]:
t0 = time.time()
zt = label_logits(model, train)
tt = time.time() - t0
lab = train["label"].to_numpy()
acc = float((zt.argmax(1) == lab).mean())
p = np.exp(zt - zt.max(1, keepdims=True)); p /= p.sum(1, keepdims=True)
stats = {"n": len(train), "teacher_acc_on_train": acc, "mean_max_prob": float(p.max(1).mean()),
         "frac_max_prob_gt_0.99": float((p.max(1) > 0.99).mean()), "val_acc_check": val_acc, "label_time_s": tt,
         "teacher_label_agreement_by_class": {c: float((zt.argmax(1)[lab == k] == k).mean()) for k, c in enumerate(["entailment", "neutral", "contradiction"])}}
print(json.dumps(stats, indent=2))

df = pd.DataFrame({"id": train["id"].to_numpy(), "label": lab, "z_ent": zt[:, 0], "z_neu": zt[:, 1], "z_con": zt[:, 2]})
df.to_csv(OUT / "teacher_logits_train_100k.csv", index=False)
json.dump({**stats, "adapter": ADAPTER, "model": MODEL, "label_token_ids": LABEL_IDS,
           "note": "teacher was trained on these same pairs (in-sample soft labels)"}, open(OUT / "teacher_stats.json", "w"), indent=2)
print("saved:", sorted(os.listdir(OUT)))
print("Next: Save Version -> Save & Run All, then make a Dataset (e.g. 'nli-teacher') from this output.")

  6,400/100,000  89 ex/s  ETA 17.6 min
  12,800/100,000  88 ex/s  ETA 16.6 min
  19,200/100,000  87 ex/s  ETA 15.4 min
  25,600/100,000  87 ex/s  ETA 14.2 min
  32,000/100,000  87 ex/s  ETA 13.1 min
  38,400/100,000  86 ex/s  ETA 11.9 min
  44,800/100,000  85 ex/s  ETA 10.8 min
  51,200/100,000  85 ex/s  ETA 9.6 min
  57,600/100,000  84 ex/s  ETA 8.4 min
  64,000/100,000  83 ex/s  ETA 7.2 min
  70,400/100,000  83 ex/s  ETA 6.0 min
  76,800/100,000  82 ex/s  ETA 4.7 min
  83,200/100,000  81 ex/s  ETA 3.5 min
  89,600/100,000  80 ex/s  ETA 2.2 min
  96,000/100,000  79 ex/s  ETA 0.8 min
{
  "n": 100000,
  "teacher_acc_on_train": 0.93277,
  "mean_max_prob": 0.9175167083740234,
  "frac_max_prob_gt_0.99": 0.2255,
  "val_acc_check": 0.9198333672017882,
  "label_time_s": 1289.4477021694183,
  "teacher_label_agreement_by_class": {
    "entailment": 0.9424626351573966,
    "neutral": 0.8923590236864254,
    "contradiction": 0.963382816014395
  }
}
saved: ['teacher_logits_train_100k.csv', 'teache